# Library: dataset loaders

Loaders for the seven open clinical datasets.

For each dataset the loader first looks in DATA_DIR (set in 00_config) and its
sub-folders for a local file. If none is found, it downloads the dataset from the
UCI Machine Learning Repository with `ucimlrepo.fetch_ucirepo` and saves it in
DATA_DIR/uci_downloads/, so the download happens only once. The file used is printed.

Accepted sources per dataset
    CKD           UCI id 336 | ckd.csv, kidney_disease.csv, chronic_kidney_disease*.arff
    Heart         UCI id 45  | heart_kmed.rda, processed.cleveland.data, heart_cleveland_upload.csv
    HeartFailure  UCI id 519 | hf.csv, heart_failure_clinical_records_dataset.csv
    BreastCancer  UCI id 17  | wdbc.data, breast_cancer_wisconsin.csv
    Cirrhosis     UCI id 878 | pbc.csv, cirrhosis.csv
    SUPPORT2      UCI id 880 | support.rda, support2.csv
    Diabetes130   UCI id 296 | readmission.rda, diabetic_data.csv

Every loader returns a Dataset with
    X      pandas DataFrame of raw features (NaN for missing; '?' is read as missing)
    y      numpy int array (1 = positive class)
    cat    list of categorical column names
    group  optional pandas Series (sex) for the subgroup analysis
    meta   dict with task, source and removed (leakage) columns
Columns known only after the outcome (follow-up time, length of stay) are removed.

Note: the manuscript's numbers were produced from the R-package copies. The UCI
versions contain the same patients and variables, but SUPPORT2 (UCI keeps the missing
values that the R copy had filled) and Diabetes 130 (converted here to the R package's
11 predictors) can give slightly different results.

## Setup
Loads `00_config` (paths) and the library notebooks from this folder.

In [26]:
# Load other notebooks in this folder as modules (all notebooks live in one folder).
import os, sys, types
import nbformat

NB_DIR = os.getcwd()   # open Jupyter in the notebooks folder


def import_notebook(name, filename):
    """Execute the code cells of a notebook in this folder into a module named `name`."""
    if name in sys.modules:
        return sys.modules[name]
    path = os.path.join(NB_DIR, filename)
    mod = types.ModuleType(name)
    mod.__file__ = path
    sys.modules[name] = mod
    for cell in nbformat.read(path, as_version=4).cells:
        if cell.cell_type == "code":
            exec(compile(cell.source, path, "exec"), mod.__dict__)
    return mod


config = import_notebook("config", "00_config.ipynb")

In [27]:
from __future__ import annotations
import os
import re
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
MISSING = ["?", "", " ", "NA", "nan", "None"]

### `Dataset` (class)

In [28]:
@dataclass
class Dataset:
    name: str
    X: pd.DataFrame
    y: np.ndarray
    cat: list
    group: pd.Series | None = None
    meta: dict = field(default_factory=dict)

### `_locate` (function)

Local file under DATA_DIR if present, otherwise download from UCI (saved for next time).

In [29]:
def _locate(dataset, *names):
    """Local file under DATA_DIR if present, otherwise download from UCI (saved for next time)."""
    uci_file = config.UCI_DATASETS[dataset][1]
    path = config.find_file(*names, uci_file)
    if path is None:
        try:
            path = config.fetch_uci(dataset)
        except ImportError:
            raise ImportError("ucimlrepo is not installed: run  %pip install ucimlrepo  and re-run") from None
        except Exception as e:
            raise FileNotFoundError(
                f"{dataset}: no local file ({', '.join(names)}) under {config.DATA_DIR}, and the UCI download "
                f"failed ({type(e).__name__}: {e}). Check the internet connection, or copy the file into DATA_DIR."
            ) from None
    print(f"[{dataset}] reading {path}")
    return path

### `_read_csv` (function)

In [30]:
def _read_csv(path, **kw):
    return pd.read_csv(path, na_values=MISSING, keep_default_na=True, **kw)

### `_col` (function)

Return the column whose name matches one of `names` (case-insensitive).

In [31]:
def _col(df, *names):
    """Return the column whose name matches one of `names` (case-insensitive)."""
    low = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in low:
            return low[n.lower()]
    raise KeyError(f"none of {names} in columns {list(df.columns)}")

### `_clean_str` (function)

In [32]:
def _clean_str(s):
    return s.astype(str).str.strip().str.replace("\t", "", regex=False) \
            .replace({"?": np.nan, "nan": np.nan, "": np.nan, "None": np.nan, "<NA>": np.nan})

### `_num` (function)

In [33]:
def _num(s):
    return pd.to_numeric(s, errors="coerce")

### `_read_rda` (function)

In [34]:
def _read_rda(path):
    import pyreadr
    return list(pyreadr.read_r(path).values())[0].reset_index(drop=True)

## CKD

In [35]:
# --------------------------------------------------------------------------- CKD
CKD_COLS = ["age", "bp", "sg", "al", "su", "rbc", "pc", "pcc", "ba", "bgr", "bu", "sc", "sod", "pot", "hemo",
            "pcv", "wbcc", "rbcc", "htn", "dm", "cad", "appet", "pe", "ane", "class"]
CKD_NUM = ["age", "bp", "sg", "al", "su", "bgr", "bu", "sc", "sod", "pot", "hemo", "pcv", "wbcc", "rbcc"]

### `_read_ckd_arff` (function)

In [36]:
def _read_ckd_arff(path):
    rows, in_data = [], False
    for line in open(path, encoding="latin-1"):
        line = line.strip()
        if not line or line.startswith("%"):
            continue
        if line.lower().startswith("@data"):
            in_data = True
            continue
        if in_data:
            tok = [t.strip() for t in line.split(",")]
            if len(tok) > 25:  # some UCI rows contain a stray empty field
                tok = [t for t in tok if t != ""]
            if len(tok) == 25:
                rows.append(tok)
    return pd.DataFrame(rows, columns=CKD_COLS)

### `load_ckd` (function)

In [37]:
def load_ckd():
    p = _locate("CKD", "ckd.csv", "kidney_disease.csv", "chronic_kidney_disease_full.arff",
                "chronic_kidney_disease.arff", "chronic_kidney_disease.csv")
    if p.lower().endswith(".arff"):
        d = _read_ckd_arff(p)
    else:
        d = pd.read_csv(p, dtype=str, keep_default_na=False)
        d.columns = [c.strip().lower() for c in d.columns]
        d = d.rename(columns={"wc": "wbcc", "rc": "rbcc", "classification": "class"})
    d = d[CKD_COLS].apply(_clean_str)
    d = d[d["class"].notna()].reset_index(drop=True)
    y = d.pop("class").str.lower().str.startswith("ckd").astype(int).values
    cat = [c for c in d.columns if c not in CKD_NUM]
    for c in CKD_NUM:
        d[c] = _num(d[c])
    return Dataset("CKD", d, y, cat, None, dict(task="CKD diagnosis", source=os.path.basename(p), dropped=[]))

## Heart (Cleveland)

In [38]:
# --------------------------------------------------------------------------- Heart (Cleveland)
HEART_COLS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope",
              "ca", "thal", "class"]

### `load_heart` (function)

In [39]:
def load_heart():
    p = _locate("Heart", "heart_kmed.rda", "heart.rda", "processed.cleveland.data", "heart_cleveland_upload.csv",
                "cleveland.csv")
    if p.lower().endswith(".rda"):
        d = _read_rda(p)
    elif p.lower().endswith(".data"):
        d = _read_csv(p, header=None, names=HEART_COLS)
    else:  # UCI id 45 (target 'num') or Kaggle version (target 'condition'/'target')
        d = _read_csv(p)
        d.columns = [c.strip().lower() for c in d.columns]
        d = d.rename(columns={"condition": "class", "num": "class", "target": "class"})
    d = d[HEART_COLS].apply(_num) if not p.lower().endswith(".rda") else d[HEART_COLS].copy()
    d = d.dropna().reset_index(drop=True)  # 303 -> 297 complete cases (ca, thal missing in 6)
    y = (d.pop("class").astype(float) > 0).astype(int).values
    for c in ["sex", "fbs", "exang"]:
        d[c] = d[c].astype(float).astype(int)
    cat = ["cp", "restecg", "slope", "thal"]
    for c in cat:
        d[c] = d[c].astype(float).astype(int).astype(str)
    grp = d["sex"].map({1: "male", 0: "female"})
    return Dataset("Heart", d, y, cat, grp, dict(task="Coronary artery disease (>0 vessels narrowed)",
                                                 source=os.path.basename(p), dropped=[]))

## Heart failure

In [40]:
# --------------------------------------------------------------------------- Heart failure
HF_COLS = ["age", "anaemia", "creatinine_phosphokinase", "diabetes", "ejection_fraction", "high_blood_pressure",
           "platelets", "serum_creatinine", "serum_sodium", "sex", "smoking"]

### `load_hf` (function)

In [41]:
def load_hf():
    p = _locate("HeartFailure", "hf.csv", "heart_failure_clinical_records_dataset.csv",
                "heart_failure_clinical_records.csv", "cardiac_failure_df.csv")
    raw = _read_csv(p)
    y = _num(raw[_col(raw, "DEATH_EVENT", "death_event")]).astype(int).values
    d = pd.DataFrame({c: _num(raw[_col(raw, c)]) for c in HF_COLS})  # 'time' (follow-up) is not used
    grp = d["sex"].map({1: "male", 0: "female"})
    return Dataset("HeartFailure", d, y, [], grp, dict(task="Death during follow-up", source=os.path.basename(p),
                                                       dropped=["time"]))

## Breast cancer

### `load_bcw` (function)

In [42]:
# --------------------------------------------------------------------------- Breast cancer
def load_bcw():
    from sklearn.datasets import load_breast_cancer
    names = list(load_breast_cancer().feature_names)
    p = config.find_file("wdbc.data", "breast_cancer_wisconsin.csv", "breast-cancer-wisconsin-diagnostic.csv",
                         "wdbc.csv", config.UCI_DATASETS["BreastCancer"][1])
    if p is None:
        try:
            p = config.fetch_uci("BreastCancer")
        except Exception as e:
            print(f"[BreastCancer] UCI download failed ({type(e).__name__}); using the copy bundled with scikit-learn")
            b = load_breast_cancer(as_frame=True)
            return Dataset("BreastCancer", b.data.copy(), (b.target == 0).astype(int).values, [], None,
                           dict(task="Malignancy", source="scikit-learn load_breast_cancer", dropped=[]))
    print(f"[BreastCancer] reading {p}")
    if p.lower().endswith(".data"):
        d = pd.read_csv(p, header=None, names=["id", "diagnosis"] + names)
    else:
        d = _read_csv(p)
        d = d.loc[:, ~d.columns.str.startswith("Unnamed")]
    y = (d.pop(_col(d, "diagnosis")).astype(str).str.strip().str.upper() == "M").astype(int).values
    d = d.drop(columns=[c for c in d.columns if c.lower() == "id"])
    if len(d.columns) == 30:  # UCI names (radius1 ... fractal_dimension3) follow the scikit-learn order
        d.columns = names
    return Dataset("BreastCancer", d.apply(_num), y, [], None,
                   dict(task="Malignancy", source=os.path.basename(p), dropped=[]))

## Cirrhosis (Mayo PBC)

### `load_pbc` (function)

In [43]:
# --------------------------------------------------------------------------- Cirrhosis (Mayo PBC)
def load_pbc():
    p = _locate("Cirrhosis", "pbc.csv", "cirrhosis.csv")
    d = _read_csv(p)
    if "Status" in d.columns or "status" not in d.columns:  # UCI id 878 / Kaggle -> Mayo PBC names and coding
        g = lambda *n: d[_col(d, *n)]
        y = (g("Status").astype(str).str.strip() == "D").astype(int).values
        yn = lambda s: s.astype(str).str.strip().map({"Y": 1, "N": 0})
        d = pd.DataFrame({
            "trt": g("Drug").astype(str).str.strip().map({"D-penicillamine": "1", "Placebo": "2"}),
            "age": _num(g("Age")) / 365.25, "sex": (g("Sex").astype(str).str.strip() == "F").astype(int),
            "ascites": yn(g("Ascites")), "hepato": yn(g("Hepatomegaly")), "spiders": yn(g("Spiders")),
            "edema": g("Edema").astype(str).str.strip().map({"N": 0.0, "S": 0.5, "Y": 1.0}),
            "bili": _num(g("Bilirubin")), "chol": _num(g("Cholesterol")), "albumin": _num(g("Albumin")),
            "copper": _num(g("Copper")), "alk.phos": _num(g("Alk_Phos")), "ast": _num(g("SGOT")),
            "trig": _num(g("Tryglicerides", "Triglycerides")), "platelet": _num(g("Platelets")),
            "protime": _num(g("Prothrombin")), "stage": _num(g("Stage"))})
    else:  # R survival::pbc
        d = d.drop(columns=[c for c in ("rownames", "id", "time") if c in d.columns])
        y = (d.pop("status") == 2).astype(int).values
        d["sex"] = (d["sex"] == "f").astype(int)
        d["trt"] = d["trt"].astype(str).replace({"nan": np.nan, "1.0": "1", "2.0": "2"})
    grp = d["sex"].map({1: "female", 0: "male"})
    return Dataset("Cirrhosis", d, y, ["trt"], grp, dict(task="Death during follow-up", source=os.path.basename(p),
                                                         dropped=["N_Days/time", "ID"]))

## SUPPORT2

In [44]:
# --------------------------------------------------------------------------- SUPPORT2
SUPPORT_FEATURES = ["age", "sex", "dzgroup", "dzclass", "num.co", "edu", "scoma", "avtisst", "race", "hday",
                    "diabetes", "dementia", "ca", "meanbp", "wblc", "hrt", "resp", "temp", "pafi", "alb", "bili",
                    "crea", "sod", "ph", "glucose", "bun", "urine", "adlp", "adlsc", "sps", "aps"]
SUPPORT_CAT = ["sex", "dzgroup", "dzclass", "race", "ca"]
# normal values recommended by the SUPPORT investigators for missing physiology
SUPPORT_NORMALS = {"alb": 3.5, "pafi": 333.3, "bili": 1.01, "crea": 1.01, "bun": 6.51, "wblc": 9.0, "urine": 2502.0}

### `load_support` (function)

In [45]:
def load_support():
    p = _locate("SUPPORT2", "support.rda", "support2.csv")
    if p.lower().endswith(".rda"):
        d = _read_rda(p)
    else:  # UCI id 880 or the Vanderbilt support2.csv
        d = _read_csv(p)
        d.columns = [c.strip().lower().replace("_", ".") for c in d.columns]
        for c in SUPPORT_FEATURES + ["death"]:
            if c not in SUPPORT_CAT:
                d[c] = _num(d[c])
        for c, v in SUPPORT_NORMALS.items():
            d[c] = d[c].fillna(v)
    d = d[d["death"].notna()].reset_index(drop=True)
    y = d["death"].astype(int).values
    d = d[SUPPORT_FEATURES].copy()
    for c in SUPPORT_CAT:
        d[c] = d[c].astype(str).str.strip().replace({"nan": np.nan, "?": np.nan, "": np.nan})
    grp = d["sex"].copy()
    return Dataset("SUPPORT2", d, y, SUPPORT_CAT, grp, dict(task="Death during follow-up", source=os.path.basename(p),
                                                            dropped=["d.time", "slos"]))

## Diabetes 130

### `_diabetic_csv_to_readmission` (function)

Convert the UCI diabetic_data.csv (or UCI id 296) to the 11-predictor format of the readmission
R package: first encounter per patient, valid sex, readmitted = '<30'.

In [46]:
# --------------------------------------------------------------------------- Diabetes 130
def _diabetic_csv_to_readmission(d):
    """Convert the UCI diabetic_data.csv (or UCI id 296) to the 11-predictor format of the readmission
    R package: first encounter per patient, valid sex, readmitted = '<30'."""
    d = d.copy()
    if "encounter_id" in d.columns:
        d = d.sort_values("encounter_id")
    if "patient_nbr" in d.columns:
        d = d.drop_duplicates("patient_nbr")
    else:
        print("[Diabetes130] warning: no patient_nbr column, so repeat encounters cannot be removed")
    d = d[d.gender.isin(["Male", "Female"])]
    src = _num(d.admission_source_id)
    payer = d.payer_code.astype(str).str.strip()
    out = pd.DataFrame({
        "readmitted": np.where(d.readmitted.astype(str).str.strip() == "<30", "Yes", "No"),
        "race": d.race.astype(str).str.strip().replace({"AfricanAmerican": "African American", "?": "Unknown",
                                                        "nan": "Unknown"}),
        "sex": d.gender, "age": d.age,
        "admission_source": np.select([src == 7, src.isin([1, 2, 3])], ["Emergency", "Referral"], "Other"),
        "blood_glucose": d.A1Cresult.astype(str).str.strip().map({"Norm": "Normal", ">7": "High", ">8": "Very High"}),
        "insurer": payer.map(lambda v: np.nan if v in ("?", "nan", "") else
                             {"MC": "Medicare", "MD": "Medicaid", "SP": "Self-Pay"}.get(v, "Private")),
        "duration": _num(d.time_in_hospital),
        "n_previous_visits": _num(d.number_outpatient) + _num(d.number_emergency) + _num(d.number_inpatient),
        "n_diagnoses": _num(d.number_diagnoses), "n_procedures": _num(d.num_procedures),
        "n_medications": _num(d.num_medications)})
    return out.reset_index(drop=True)

### `load_readmission` (function)

In [47]:
def load_readmission():
    p = _locate("Diabetes130", "readmission.rda", "diabetic_data.csv")
    if p.lower().endswith(".rda"):
        d = _read_rda(p)
    else:
        d = _diabetic_csv_to_readmission(pd.read_csv(p, keep_default_na=True, low_memory=False))
    y = (d.pop("readmitted").astype(str) == "Yes").astype(int).values
    cat = ["race", "sex", "admission_source", "blood_glucose", "insurer"]
    d["age"] = d["age"].astype(str).map(lambda s: float(re.findall(r"\d+", s)[0]) + 5 if re.findall(r"\d+", s) else np.nan)
    for c in cat:
        d[c] = d[c].astype(str).replace({"nan": np.nan})
    grp = d["sex"].copy()
    return Dataset("Diabetes130", d, y, cat, grp, dict(task="30-day readmission", source=os.path.basename(p),
                                                       dropped=[]))

In [48]:
LOADERS = dict(CKD=load_ckd, Heart=load_heart, HeartFailure=load_hf, BreastCancer=load_bcw,
               Cirrhosis=load_pbc, SUPPORT2=load_support, Diabetes130=load_readmission)

### `load` (function)

In [49]:
def load(name):
    return LOADERS[name]()

## Quick check

In [50]:
if __name__ == "__main__":
    for n in LOADERS:
        d = load(n)
        print(f"   {n}: {d.X.shape}, prevalence {d.y.mean():.3f}, missing {d.X.isna().mean().mean():.3f}")

[CKD] reading D:\New_Research_2027\Clinical_XAI_Faithfulness\uci_downloads\uci_336_chronic_kidney_disease.csv
   CKD: (400, 24), prevalence 0.625, missing 0.105
[Heart] reading D:\New_Research_2027\Clinical_XAI_Faithfulness\heart+disease\processed.cleveland.data
   Heart: (297, 13), prevalence 0.461, missing 0.000
[HeartFailure] reading D:\New_Research_2027\Clinical_XAI_Faithfulness\heart+failure+clinical+records\heart_failure_clinical_records_dataset.csv
   HeartFailure: (299, 11), prevalence 0.321, missing 0.000
[BreastCancer] reading D:\New_Research_2027\Clinical_XAI_Faithfulness\breast+cancer+wisconsin+diagnostic\wdbc.data
   BreastCancer: (569, 30), prevalence 0.373, missing 0.000
[Cirrhosis] reading D:\New_Research_2027\Clinical_XAI_Faithfulness\cirrhosis+patient+survival+prediction+dataset-1\cirrhosis.csv
   Cirrhosis: (418, 17), prevalence 0.385, missing 0.145
[SUPPORT2] reading D:\New_Research_2027\Clinical_XAI_Faithfulness\uci_downloads\uci_880_support2.csv
   SUPPORT2: (9105